In [1]:
import pandas as pd

In [2]:
data = pd.read_csv("Datasets/Restaurant_Reviews.tsv", delimiter="\t")
data.head()

,Review,Liked
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [3]:
data.shape

(1000, 2)

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Review  1000 non-null   object
 1   Liked   1000 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 15.8+ KB


In [5]:
data['Liked'].value_counts()

Liked
1    500
0    500
Name: count, dtype: int64

In [6]:
corpus = data['Review']

In [7]:
import spacy

In [9]:
nlp = spacy.load("en_core_web_sm")

In [10]:
text = corpus[0]

In [11]:
text

'Wow... Loved this place.'

In [12]:
doc = nlp(corpus[0])

In [13]:
for token in doc:
    print(token)

Wow
...
Loved
this
place
.


In [16]:
import re

In [18]:
pattern = r'[^A-Za-z]'
re.findall(pattern, text)

['.', '.', '.', ' ', ' ', ' ', '.']

In [19]:
text = re.sub(pattern, ' ', text)

In [20]:
text

'Wow    Loved this place '

In [21]:
doc = nlp(text)

In [22]:
for token in doc:
    token = str(token)
    if not token.isspace():
        print(token)

Wow
Loved
this
place


In [23]:
corpus.shape

(1000,)

In [24]:
corpus.shape[0]

1000

In [26]:
stop_words = nlp.Defaults.stop_words
print(stop_words)

{'every', 'can', 'sixty', 'something', 'your', 'throughout', 'what', 'either', 'ever', 'from', "'s", 'moreover', 'must', 'should', 'it', 'once', 'over', 'well', '’m', '‘ve', 'become', 'afterwards', 'next', 'rather', 'her', '’re', 'besides', 'myself', 'former', 'was', 'therefore', 'make', 'above', 'fifteen', 'none', 'still', 'therein', 'otherwise', 'whose', 'these', 'already', 'n’t', 'else', 'whereas', 'whenever', 'nothing', 'they', 'sometimes', 'although', 'themselves', 'down', '‘d', 'those', 'three', 'yourselves', 'himself', 'whither', 'keep', 'regarding', 'up', 'with', 'yourself', 'hers', 'very', 'thence', 'an', 'bottom', 'not', '‘ll', 'namely', 'part', 'becoming', '’d', 'really', 'indeed', 'the', 'back', 'ca', 'neither', 'however', 'anywhere', 'my', 'used', 'forty', 'amount', 'due', 'will', 'ourselves', 'you', 'whoever', 'see', 'most', 'even', '’s', 'move', 'made', 'least', 'show', 'say', 'less', 'mostly', "'ll", 'upon', 'when', 'on', 'nowhere', 'serious', 'him', 'go', 'why', 'beyon

In [27]:
'not' in stop_words

True

In [28]:
'never' in stop_words

True

In [29]:
type(stop_words)

set

In [30]:
stop_words.discard('not')
stop_words.discard('never')

In [31]:
'not' in stop_words

False

In [34]:
pattern = r'[^A-Za-z]'
processed_text = []            #list of processed reviews
for i in range(corpus.shape[0]):     # execute loop till row count
    text = corpus[i]           # fetch row
    result = []                # list of words of particular reviews
    text = re.sub(pattern, ' ', text)
    doc = nlp(text)
    for token in doc:
        if token.lemma_ not in stop_words:
            result.append(token.lemma_)       # append processed word in list
    result = ' '.join(result)                  #  join list of words(particular review) in string
    processed_text.append(result)          # add processed reviews in a list of review

In [35]:
from sklearn.feature_extraction.text import CountVectorizer

In [36]:
obj = CountVectorizer()
result = obj.fit_transform(processed_text)
result

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 4768 stored elements and shape (1000, 1536)>

In [37]:
result = result.toarray()
result

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(1000, 1536))

In [41]:
obj.get_feature_names_out()

array(['absolute', 'absolutely', 'absolutley', ..., 'yum', 'yummy',
       'zero'], shape=(1536,), dtype=object)

In [43]:
X = result
y = data['Liked']

In [44]:
from sklearn.model_selection import train_test_split

In [45]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state=40)

In [46]:
from sklearn.naive_bayes import GaussianNB
model = GaussianNB()
model.fit(X_train, y_train)

,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",1e-09


In [47]:
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

In [49]:
from sklearn.metrics import accuracy_score

In [50]:
train_acc = accuracy_score(y_train, train_pred)
test_acc = accuracy_score(y_test, test_pred)

In [51]:
print('Training Accuracy : ' , train_acc)
print('Testing Accuracy : ' , test_acc)

Training Accuracy :  0.9471428571428572
Testing Accuracy :  0.73


In [63]:
new_text = "I like Pavbhaji"

# cleaning
text = re.sub(pattern, ' ', new_text)

# spacy
doc = nlp(text)

result = []
for token in doc:
    if token.lemma_ not in stop_words:
        result.append(token.lemma_)

processed = ' '.join(result)

# vectorize
x_new = obj.transform([processed]).toarray()
 
# predict
pred = model.predict(x_new)

# if pred[0] == 1:
#     print("Positive Review")
# else:
#     print("Negative Review")

(1, 2)


ValueError: X has 2 features, but GaussianNB is expecting 1536 features as input.

In [61]:
#pred = model.predict(result)